In [5]:
import time
from math import sqrt

## Compréhension de listes (_list comprehension_)

On souhaite créer une liste contenant les carrés ($f(x) = x^2$) des $n$ premiers nombres entiers.

Voici la méthode naïve avec une boucle `for` :

In [6]:
n = 9_000_000
n2 = int(sqrt(n))

In [7]:
%%timeit

my_list = list()
for i in range(n):
    my_list.append(i*i)

456 ms ± 7.64 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


On peut faire la même chose avec les compréhensions de liste:

In [9]:
%%timeit
my_list = [i * i for i in range(n)]

401 ms ± 20.4 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


On fait maintenant des boucles `for` imbriquées :

In [10]:
%%timeit

my_list = list()
for j in range(10_000):
    for i in range(10_000):
        my_list.append(i*j)

5.09 s ± 154 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%%timeit
my_list = [i * j for j in range(10_000) for i in range(10_000)]

4.45 s ± 160 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Si on rajoute des `if` / `else` dans la boucle `for` :

In [12]:
%%timeit

my_list = list()
for i in range(n):
    if i%2 == 0:
        my_list.append(i*i)
    else:
        my_list.append(i+i)

615 ms ± 41.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
%%timeit

my_list2 = [i * i if i % 2 == 0 else i + i for i in range(n)]

525 ms ± 5.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


Vérifions que les deux listes sont bien égales (il faut les recalculer car `timeit` ne permet pas de les réutiliser dans d'autres cellules) :

In [14]:
my_list = list()
for j in range(10_000):
    for i in range(10_000):
        my_list.append(i*j)

my_list2 = [i * j for j in range(10_000) for i in range(10_000)]

my_list == my_list2

True

On peut tester des conditions plus complexes. 

**Remarque :** pour obtenir la compréhension de liste, il suffit d'écrire les boucles dans le sens où on l'ecrirait dans la methode naïve.

In [15]:
%%timeit

my_list = list()
for j in range(10_000):
    if j % 2 == 0:
        for i in range(10_000):
            if i > j:
                my_list.append(i*j)
            else:
                my_list.append("Rien")

2.53 s ± 55.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [1]:
%%timeit
my_list2 = [i * j if i > j else "Rien" for j in range(10_000) if j % 2 == 0 for i in range(10_000)]

2.12 s ± 13.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [2]:
my_list = list()
for j in range(10_000):
    if j % 2 == 0:
        for i in range(10_000):
            if i > j:
                my_list.append(i*j)
            else:
                my_list.append("Rien")

my_list2 = [i * j if i > j else "Rien" for j in range(10_000) if j % 2 == 0 for i in range(10_000)]

my_list == my_list2

True

Remarques : 
- si on veut filtrer une liste avec uniquement un `if`, on le met après le `for` :
`[i for i in ma_list if i > 0]`
 
- si on veut utiliser un `else`, il faut le placer avant le `for` :
`[i if i > 0 else 0 for i in ma_liste]`

# Les générateurs

In [3]:
import numpy as np

data = np.random.randint(100, size=100_000)

In [4]:
def apply_generator(generator):
    sum_batch = 0
    sum_elem = 0
    for batch in generator(data):
        if sum_elem == 0:
            print(f'Valeur du premier batch {batch}')
        sum_elem += len(batch) if isinstance(batch, np.ndarray) else 1
        sum_batch += 1
    print(f'Echantillons délivrés par le générateurs : {sum_elem}')
    print(f'Nombre de batches : {sum_batch}')

Faites un générateur qui retourne les échantillons un par un :

In [5]:
def my_generator(data):
    for x in data:
        yield x

In [6]:
apply_generator(my_generator)

Valeur du premier batch 50
Echantillons délivrés par le générateurs : 100000
Nombre de batches : 100000


Faites un générateur qui retourne des batchs de 10 échantillons :

In [7]:
def my_generator(data):
    for i in range(0, len(data), 10):
        yield data[i:i+10]

In [8]:
apply_generator(my_generator)

Valeur du premier batch [50 79 47 75 55 74 35 44 70 43]
Echantillons délivrés par le générateurs : 100000
Nombre de batches : 10000


Faites un générateur qui retourne des batchs de 10 échantillons normalisés (normalisation sur les données du batch)

In [9]:
def my_generator(data):
    for i in range(0, len(data), 10):
        batch = data[i:i + 10]
        mean = batch.mean()
        std = batch.std()
        if std == 0:
            yield np.zeros_like(batch, dtype=float)
        else:
            yield (batch - mean) / std

In [10]:
apply_generator(my_generator)

Valeur du premier batch [-0.47813361  1.44768232 -0.67735595  1.18205253 -0.14609638  1.11564509
 -1.47424529 -0.87657828  0.8500153  -0.94298573]
Echantillons délivrés par le générateurs : 100000
Nombre de batches : 10000
